# Лабораторная работа №1 — продолжение
## Первичное исследование и оценка качества данных

**Датасет:** `hotel_bookings.csv` (Hotel Booking Demand)

Ниже выполнены задания 1–8 из продолжения лабораторной работы.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

# Ноутбук запускается из каталога notebooks/
df = pd.read_csv("../data/hotel_bookings.csv")

print("Размер датасета:", df.shape)
display(df.head())


## Задание 1. Типы признаков и шкалы измерения

Для каждого признака определим тип шкалы.

- **Номинальная:** категории без естественного порядка.
- **Порядковая:** категории с естественным порядком.
- **Интервальная:** разности между значениями интерпретируемы, но ноль не является абсолютным отсутствием.
- **Шкала отношений / абсолютная:** есть естественный ноль, поэтому осмыслены отношения величин.

В данном датасете большинство категориальных признаков являются номинальными, а количественные показатели — признаками шкалы отношений.


In [ ]:
scale_map = {
    "hotel": "номинальная",
    "is_canceled": "номинальная (бинарная)",
    "lead_time": "отношений / абсолютная",
    "arrival_date_year": "интервальная (год как календарная шкала)",
    "arrival_date_month": "номинальная",
    "arrival_date_week_number": "отношений / абсолютная",
    "arrival_date_day_of_month": "отношений / абсолютная",
    "stays_in_weekend_nights": "отношений / абсолютная",
    "stays_in_week_nights": "отношений / абсолютная",
    "adults": "отношений / абсолютная",
    "children": "отношений / абсолютная",
    "babies": "отношений / абсолютная",
    "meal": "номинальная",
    "country": "номинальная",
    "market_segment": "номинальная",
    "distribution_channel": "номинальная",
    "is_repeated_guest": "номинальная (бинарная)",
    "previous_cancellations": "отношений / абсолютная",
    "previous_bookings_not_canceled": "отношений / абсолютная",
    "reserved_room_type": "номинальная",
    "assigned_room_type": "номинальная",
    "booking_changes": "отношений / абсолютная",
    "deposit_type": "номинальная",
    "agent": "номинальная (идентификатор)",
    "company": "номинальная (идентификатор)",
    "days_in_waiting_list": "отношений / абсолютная",
    "customer_type": "номинальная",
    "adr": "отношений / абсолютная",
    "required_car_parking_spaces": "отношений / абсолютная",
    "total_of_special_requests": "отношений / абсолютная",
    "reservation_status": "номинальная",
    "reservation_status_date": "интервальная (дата)"
}

scale_table = pd.DataFrame({
    "Признак": df.columns,
    "Шкала": [scale_map.get(c, "требует отдельного обоснования") for c in df.columns]
})
display(scale_table)


### Сравнение двух признаков

Возьмём `hotel` (номинальная шкала) и `adr` (шкала отношений).

Для `hotel`:
- среднее — методологически неверно;
- медиана — методологически неверна;
- мода — корректна.

Для `adr`:
- среднее — корректно;
- медиана — корректно;
- мода — математически допустима, но для непрерывного показателя обычно малоинформативна.

То, что `pandas` способен посчитать числовой агрегат после кодирования категорий, ещё не означает, что такой агрегат имеет содержательный смысл.


In [ ]:
print("Мода hotel:")
display(df["hotel"].mode())

print("\nСтатистики adr:")
display(df["adr"].describe()[["mean", "50%", "min", "max"]])


## Задание 2. Пропуски и их природа: MCAR / MAR / MNAR

Для анализа выберем `agent`, поскольку в нём есть значительное количество пропусков.

Гипотезы:

1. **MCAR:** вероятность пропуска не зависит ни от наблюдаемых, ни от ненаблюдаемых данных.
2. **MAR:** вероятность пропуска связана с другими наблюдаемыми признаками, например типом отеля, каналом распределения или сегментом рынка.
3. **MNAR:** вероятность пропуска зависит от самого ненаблюдаемого значения или от причины, которую нельзя объяснить доступными наблюдаемыми признаками.

Полностью доказать MNAR только по наблюдаемым данным нельзя. Поэтому практический анализ позволяет искать свидетельства против MCAR и проверять правдоподобие MAR, но не строго доказать MNAR.


In [ ]:
missing_agent = df["agent"].isna()

print("Пропусков agent:", missing_agent.sum())
print("Доля пропусков:", round(missing_agent.mean() * 100, 2), "%")

# Проверка зависимости пропуска от наблюдаемых категориальных признаков
for col in ["hotel", "market_segment", "distribution_channel", "deposit_type", "customer_type"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        tmp = df.groupby(col)["agent"].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
        display(tmp.to_frame("Доля пропусков agent"))


In [ ]:
# Числовые признаки: сравнение распределений между строками с agent и без agent
numeric_cols = [
    "lead_time", "adr", "stays_in_week_nights",
    "stays_in_weekend_nights", "total_of_special_requests"
]

comparison = []
for col in numeric_cols:
    if col in df.columns:
        comparison.append({
            "Признак": col,
            "Среднее при agent=NaN": df.loc[missing_agent, col].mean(),
            "Среднее при agent заполнен": df.loc[~missing_agent, col].mean(),
            "Медиана при agent=NaN": df.loc[missing_agent, col].median(),
            "Медиана при agent заполнен": df.loc[~missing_agent, col].median()
        })

display(pd.DataFrame(comparison))


**Интерпретация:** если доля пропусков `agent` заметно различается между наблюдаемыми группами, это является аргументом против MCAR и поддерживает возможность MAR. Для MNAR по одному такому анализу вывода делать нельзя: требуется дополнительная информация о механизме формирования данных.



## Задание 3. Индикатор пропуска как источник информации

Создадим бинарный признак `agent_missing`.

Сам факт пропуска может быть информативным, если пропуск возникает не случайно: например, определённый тип бронирования или канал продаж чаще не содержит идентификатор агента.

Значимость индикатора в модели может означать, что импутация одного значения скрыла различия между строками с исходным значением и строками с пропуском. В таком случае индикатор помогает модели восстановить информацию о самом механизме пропуска.

Примеры:
- **полезен:** `agent_missing`, если отсутствие агента связано с типом бронирования;
- **бесполезен:** пропуск возник из-за случайного технического сбоя;
- **опасен:** пропуск напрямую кодирует утечку целевой переменной или постфактум известную информацию.


In [ ]:
df["agent_missing"] = df["agent"].isna().astype(int)

display(
    df.groupby("agent_missing")["is_canceled"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "Доля отмен"})
)


## Задание 4. Выброс — ошибка или сигнал?

Выберем числовой признак `adr` (average daily rate).

В исходном исследовании для `adr` были обнаружены выбросы по правилу IQR, а также отрицательное значение `adr`. citeturn0view0

Возможные причины:
- ошибка ввода;
- ошибка преобразования/загрузки данных;
- специальная корректировка стоимости;
- редкая, но реальная экономическая ситуация.

Удалять выброс автоматически не следует. Если значение невозможно физически или является технической ошибкой, его можно исправить/удалить. Если это редкое, но реальное наблюдение, удаление исказит распределение.

Преобразование (`log1p`) полезно, когда проблема заключается в сильной правосторонней асимметрии, а не в ошибочном значении.


In [ ]:
adr = df["adr"].dropna()

q1 = adr.quantile(0.25)
q3 = adr.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df[(df["adr"] < lower) | (df["adr"] > upper)]

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Нижняя граница:", lower)
print("Верхняя граница:", upper)
print("Количество выбросов по IQR:", len(outliers))
print("Количество отрицательных adr:", (df["adr"] < 0).sum())

display(df.loc[df["adr"] < 0, ["hotel", "adr", "is_canceled", "reservation_status"]].head())


## Задание 5. Преобразование или винзоризация?

Для сильно асимметричного `adr` сравним три подхода.

- **Логарифмирование:** хорошо подходит для положительных сильно скошенных величин; уменьшает влияние крупных значений и часто улучшает линейные модели.
- **Корень:** более мягкое преобразование, чем логарифм; подходит, когда нужна умеренная коррекция асимметрии.
- **Винзоризация:** заменяет экстремальные значения границами; сохраняет число наблюдений, но непосредственно меняет хвост распределения.

Логарифм и корень меняют шкалу признака, поэтому интерпретация коэффициентов модели меняется. Винзоризация сохраняет исходную шкалу, но искусственно ограничивает экстремальные значения.


In [ ]:
positive_adr = df["adr"].where(df["adr"] > 0)

transformed = pd.DataFrame({
    "adr": df["adr"],
    "log1p_adr": np.log1p(df["adr"].clip(lower=0)),
    "sqrt_adr": np.sqrt(df["adr"].clip(lower=0))
})

# Винзоризация по 1-му и 99-му процентилям
low, high = df["adr"].quantile([0.01, 0.99])
transformed["winsorized_adr"] = df["adr"].clip(lower=low, upper=high)

display(transformed.describe().T[["mean", "50%", "std", "min", "max"]])


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df["adr"].dropna(), bins=80)
ax.set_title("Исходное распределение adr")
ax.set_xlabel("adr")
ax.set_ylabel("Количество")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(transformed["log1p_adr"].dropna(), bins=80)
ax.set_title("Распределение log1p(adr)")
ax.set_xlabel("log1p(adr)")
ax.set_ylabel("Количество")
plt.show()


## Задание 6. Средние, которые вводят в заблуждение

Для `adr` сравним арифметическое, медиану и геометрическое среднее.

Геометрическое среднее применимо только к положительным значениям, поэтому нулевые и отрицательные значения исключаем.

Если распределение имеет длинный правый хвост, арифметическое среднее увеличивается из-за крупных значений. Медиана показывает центральное наблюдение и обычно лучше описывает «типичный» заказ при сильной асимметрии. Геометрическое среднее находится между ними по характеру воздействия крупных значений, но имеет другую интерпретацию.


In [ ]:
adr_valid = df.loc[df["adr"] > 0, "adr"]

arithmetic_mean = adr_valid.mean()
median = adr_valid.median()
geometric_mean = np.exp(np.log(adr_valid).mean())

print(f"Арифметическое среднее: {arithmetic_mean:.2f}")
print(f"Медиана: {median:.2f}")
print(f"Геометрическое среднее: {geometric_mean:.2f}")


## Задание 7. Неправильная диаграмма

Для `adr` по типам отеля можно использовать boxplot или histogram.

Для этих данных плохим выбором была бы круговая диаграмма для большого количества отдельных значений `adr`: она не показывает форму распределения, хвосты и выбросы.

Неверный вывод зрителя мог бы состоять в том, что несколько секторов представляют «средние цены», хотя круговая диаграмма плохо передаёт распределение количественного признака.

Для количественного `adr` лучше использовать histogram, boxplot или violin plot.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
df.boxplot(column="adr", by="hotel", ax=ax)
plt.suptitle("")
ax.set_title("Распределение adr по типу отеля")
ax.set_xlabel("Тип отеля")
ax.set_ylabel("adr")
plt.show()


## Задание 8. Одна информация — разные графики

Рассмотрим пару признаков `lead_time` и `adr`.

**График 1 — scatter plot:** показывает отдельные наблюдения, плотность точек, выбросы и возможную связь между признаками.

**График 2 — hexbin:** агрегирует точки по двумерным ячейкам и хорошо показывает области высокой концентрации наблюдений, когда scatter plot становится слишком плотным.

Scatter plot легче использовать для поиска отдельных выбросов и локальных групп. Hexbin удобнее для оценки общей структуры плотности при большом количестве наблюдений.


In [ ]:
sample = df[["lead_time", "adr"]].dropna().sample(
    min(10000, df[["lead_time", "adr"]].dropna().shape[0]),
    random_state=42
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(sample["lead_time"], sample["adr"], alpha=0.25)
ax.set_title("lead_time и adr — scatter plot")
ax.set_xlabel("lead_time")
ax.set_ylabel("adr")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
hb = ax.hexbin(sample["lead_time"], sample["adr"], gridsize=35, mincnt=1)
ax.set_title("lead_time и adr — hexbin")
ax.set_xlabel("lead_time")
ax.set_ylabel("adr")
plt.colorbar(hb, ax=ax, label="Количество наблюдений")
plt.show()


# Итог

В работе:
1. классифицированы признаки по шкалам измерения;
2. рассмотрен механизм пропусков `agent` через гипотезы MCAR/MAR/MNAR;
3. создан индикатор пропуска;
4. исследованы выбросы `adr`;
5. сравнены логарифмирование, корень и винзоризация;
6. сравнены арифметическое, медиана и геометрическое среднее;
7. разобран пример неудачной визуализации;
8. построены два разных графика для пары `lead_time` и `adr`.

Все выводы следует интерпретировать с учётом природы исходного датасета и качества данных.
